In [22]:
%%writefile snake_race.py

## THIS IS JUST A PRECURSOR. DO NOT RUN THE CODE. JUST READ THE COMMENTS FOR YOUR UNDERSTANDING

from turtle import Turtle,Screen
import time

screen = Screen()
screen.setup(width=600,height=600)

#tracer method is used to turn off all animation.
screen.tracer(0)

#appending the 3 turtle objects to this list
segments = []

#setting x,y position for the second and third turtles
start_x = 0
start_y = 0

#creating 3 turtle object
for i in range(0,3):
    t = Turtle(shape='square')
    t.penup()
    t.setpos(start_x,start_y)
    #setting the second and third turtles behind each other by 20 pixels
    start_x -= 20
    #appending to the list
    segments.append(t)

# loop_num = 0
# while loop_num < 45:
game_on = True
while game_on == True:
    #this is to specify a 0.5 second delay between each movement. this way, you're controlling the speed of the snake
    time.sleep(0.5)
    
    #screen.update is used to turn on the animations after you've turned them off with the tracer method. 
    #this way you can control what the user sees and skip the intervening steps when you have a complex drawing.
    
    #if you have screen.update within the for loop you will see how each segment moves individually, one after the other.
    #obviously, you do not want this. You want to see a "snake". Hence, you move the screen.update() method **outside** the for
    #loop. This way you will see them move all at once, giving the impression of a snake.
   
    #first, you will see all 3 segments together ala snake.
    screen.update()   
    
    #then the first loop will run and all 3 segments will forward, but you will see them as one.
    #for seg_num in segments:
    #   seg_num.forward(20)
    #loop_num += 1
    
    #here, you moving the 3rd segment into the 2nd segment position and the 2nd to the 1st position
    for seg_num in range(len(segments)-1,0,-1): #range(start,stop,step) - starting from the last segment 
        new_x = segments[seg_num - 1].xcor()#getting the xcor of the the segment in front
        new_y = segments[seg_num - 1].ycor()#getting the xcor of the the segment in front
        segments[seg_num].goto(new_x,new_y)#moving the segment
    
    #this is to move the 1st segment. Otherwise, all segments will end up in the same place.
    segments[0].forward(20)
    segments[0].left(90)
    
screen.mainloop()
  
!python snake_race.py

Overwriting snake_race.py


***

In [1]:
%%writefile snake_class.py

# class to create the snake
#everything to do with the snake i.e appearance and movement will be controlled in this class

from turtle import Turtle,Screen
import time

screen = Screen()

#constants to control snake movement

STARTING_POSITIONS = [(0, 0), (-20, 0), (-40, 0)]
MOVE_DISTANCE = 20

UP = 90
DOWN = 270
RIGHT = 0
LEFT = 180

class Snake:
    def __init__(self):
        
        self.segments = []
        self.create_snake()
        
    def create_snake(self):
        for position in STARTING_POSITIONS:
            self.add_segment(position)
            
    
    #the position to tell it where to add the segment
    def add_segment(self,position):
            new_segment = Turtle(shape='square')
            new_segment.penup()
            new_segment.goto(position)
            self.segments.append(new_segment)        
        
    #add a new segment to the snake when it collides with the food    
    def extend(self):
        self.add_segment(self.segments[-1].position())

        
    def move_snake(self):
        for seg_num in range(len(self.segments)-1,0,-1):
            new_x = self.segments[seg_num - 1].xcor()
            new_y = self.segments[seg_num - 1].ycor()
            self.segments[seg_num].goto(new_x,new_y)
                
        self.segments[0].forward(MOVE_DISTANCE)
        #self.segments[0].left(90)
        
    def reset(self):
        for seg in self.segments:
            seg.goto(1000,1000)
        self.segments.clear()
        self.create_snake()

    
    #also, you are controlling the movement of the snake by controlling the head of the snake.
    #segments[0] is the head of the snake.
    #move_snake ensures the rest of the body follows the head.
    
    #the snake is not allowed to move back on itself. Hence, you need to check which direction it is going first.
    #if it going DOWN, then it cannot move UP. You need to turn right twice or left twice to head back UP again.
    #similalry, for LEFT and RIGHT as well.
    
    def move_left(self):
        if self.segments[0].heading() != RIGHT:
            self.segments[0].setheading(LEFT)
        
    def move_right(self):
        if self.segments[0].heading() != LEFT:
            self.segments[0].setheading(RIGHT)
        
    def move_up(self):
        if self.segments[0].heading() != DOWN:
            self.segments[0].setheading(UP)
        
    def move_down(self):
        if self.segments[0].heading() != UP:
            self.segments[0].setheading(DOWN)       

             

Overwriting snake_class.py


In [2]:
%%writefile food_class.py

from turtle import Turtle
import random

class Food(Turtle):
    
    def __init__(self):
        super().__init__()
        self.shape(name='circle')
        self.penup() #so that it doesn't draw everytime it moves
        self.fillcolor('blue')
        self.shapesize(stretch_wid=0.5, stretch_len=0.5)
        self.speed('fastest') #this way, the food just disappears and reappears and you do not see the intervening animation
        #of it actually moving
        self.refresh()
        
        #screensize is 600 x 600.
        #hence x ranges from -300 to 300. Similarly for y
        #you do not want the food to appear right at the edge of the screen since it will collide with the 'wall'
        #hence, keeping the limits at 280 to keep the snake away from the wall
    
    #refresh method is to get the food object to goto a new location when the snake collides it during the game    
    def refresh(self):
        random_x = random.randint(-280,280)
        random_y = random.randint(-280,280)
        
        self.goto(random_x,random_y)
        

Overwriting food_class.py


In [6]:
%%writefile scoreboard_class.py

from turtle import Turtle

#create high_score file to store the high score
#data.txt was created before with the starting high score set at 0
with open('data.txt') as file:
    contents = file.read()

#convert the string into an integer to use in the score class
HIGH_SCORE = int(contents)


class Score(Turtle):
    
    def __init__(self):
        super().__init__()
        self.score = 0
        #setting the high_score initially to 0
        self.high_score = HIGH_SCORE
        self.penup()
        self.goto(x=0,y=270)
        self.hideturtle()
        self.write(arg=f"Score: {self.score} High Score: {self.high_score}",align = "center", font = ("Arial", 20, "normal"))  
    
    def increase_score(self):
        self.score += 1
        self.clear()
        self.write(arg=f"Score: {self.score} High Score: {self.high_score}",align = "center", font = ("Arial", 20, "normal"))
        
    def update_score(self):
        self.clear()
        self.write(arg=f"Score: {self.score} High Score: {self.high_score}",align = "center", font = ("Arial", 20, "normal"))  
        
    def reset(self):
    #if the current score is > than the high score, then change the high score to the current score
        if self.score > self.high_score:
            
            #updating the data file with the new high score. whenever you start a new game, the last high score will be displayed
            with open('data.txt', mode='w') as file:
                file.write(f'{self.score}')
                
            #setting the high score in the current game to the latest score. whenever you die, the latest high score will be
            #displayed
            with open('data.txt') as file:
                self.high_score = int(file.read())
                
        self.score = 0
        self.update_score()         
        
#     def game_over(self):
#         self.goto(0,0)
#         self.write(arg="GAME OVER",align = "center", font = ("Arial", 20, "normal"))
        
        
    

Overwriting scoreboard_class.py


In [7]:
%%writefile main.py

#bringing everyhting together here

from turtle import Screen
from snake_class import Snake
from food_class import Food
from scoreboard_class import Score
import time

screen = Screen()
screen.setup(width=600,height=600)
screen.tracer(0)

snake = Snake()#creating a snake object
food = Food()#creating a food object
score = Score()#creating a score object

#the screen events are to be placed outside the while loop.
#if you place them inside the while loop, for each loop the the functions are rebounded & 
#re-run and you won't be able to control the snake properly.
#hence, you always bind outside the loop and let the game loop handle the movement.

screen.listen()
screen.onkey(snake.move_left,"Left")
screen.onkey(snake.move_right,"Right")
screen.onkey(snake.move_up,"Up")
screen.onkey(snake.move_down,"Down")

game_on = True

while game_on:
    snake.move_snake()
    
    #screen will update every 0.5 secs and the snake will move forward by 20 pixels
    screen.update()
    time.sleep(0.1)
    
    #detect collision with food
    if snake.segments[0].distance(food) < 15:
        #you know that the size of the food is 10 x 10. You give yourself a buffer i.e. 15 pixels. 
        # when the snake is that close, you know it will collide with the food.
        food.refresh()
        score.increase_score()
        snake.extend()
        
    #detect collision with wall
    if snake.segments[0].xcor() > 260 or snake.segments[0].xcor() < -260 or snake.segments[0].ycor() > 260 or snake.segments[0].ycor() < -260:
        #game_on = False
        #score.game_over()
        score.reset()
        snake.reset()

        
    #detect collision with tail
    #if the head collides with the tail, trigger game_over sequence
    for segment in snake.segments[1:]:
        if snake.segments[0].distance(segment) < 10:
            #game_over = False
            #score.game_over()
            score.reset()
            snake.reset()
           


    
screen.mainloop() 

Overwriting main.py


In [10]:
!python main.py